# Step 7 — Semantic Search

Notebook ini melakukan semantic search menggunakan embedding yang telah
dibuat pada Step 6.

Alur:

1. Load model embedding.
2. Buat embedding untuk query pengguna.
3. Ambil embedding dari SQLite.
4. Hitung cosine similarity.
5. Tampilkan Top-K chunk paling relevan.

> Pada tahap ini kita belum menggunakan `sqlite-vec`. Perhitungan similarity
dilakukan di Python agar mudah dipahami.


In [1]:
from pathlib import Path
import sqlite3
import numpy as np
from sentence_transformers import SentenceTransformer

DB_PATH = Path("../data/linux_docs.db")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

MODEL_NAME = "BAAI/bge-small-en-v1.5"
model = SentenceTransformer(MODEL_NAME)

print("Model:", MODEL_NAME)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model: BAAI/bge-small-en-v1.5


## Masukkan query

In [2]:
query = "How do I partition the disk during Arch Linux installation?"
print(query)


How do I partition the disk during Arch Linux installation?


In [3]:
query_embedding = model.encode(
    query,
    normalize_embeddings=True
).astype(np.float32)

print("Embedding dimension:", query_embedding.shape[0])


Embedding dimension: 384


## Load embeddings dari SQLite

In [4]:
cursor.execute('''
SELECT
    e.chunk_id,
    e.embedding,
    c.section,
    c.content
FROM embeddings e
JOIN chunks c
ON e.chunk_id = c.id
''')

rows = cursor.fetchall()
print("Embeddings:", len(rows))


Embeddings: 41


## Hitung cosine similarity

In [5]:
results = []

for row in rows:

    embedding = np.frombuffer(
        row["embedding"],
        dtype=np.float32
    )

    score = float(np.dot(query_embedding, embedding))

    results.append({
        "chunk_id": row["chunk_id"],
        "section": row["section"],
        "score": score,
        "content": row["content"]
    })

results = sorted(
    results,
    key=lambda x: x["score"],
    reverse=True
)


## Top 5 hasil

In [6]:
TOP_K = 5

for i, item in enumerate(results[:TOP_K], start=1):

    print("=" * 100)
    print(f"Rank    : {i}")
    print(f"Score   : {item['score']:.4f}")
    print(f"Section : {item['section']}")
    print()
    print(item["content"][:700])
    print()


Rank    : 1
Score   : 0.7738
Section : Introduction

This document is a guide for installing [Arch Linux](/title/Arch_Linux) using the live system booted from an installation medium made from an official installation image. Its accessibility features are described on the page [Install Arch Linux with accessibility options](/title/Install_Arch_Linux_with_accessibility_options). For alternative means of installation, see [Category:Installation process](/title/Category:Installation_process).

Before installing, it would be advised to view the [FAQ](/title/FAQ). For conventions used in this document, see [Help:Reading](/title/Help:Reading). In particular, code examples may contain placeholders (formatted in *italics*

This guide is kept concise an

Rank    : 2
Score   : 0.7717
Section : Boot loader

Choose a boot loader applicable to your partitioning scheme and install it. See the explanations and the comparison table in [Arch boot process#Boot loader](/title/Arch_boot_process#Boot_loader

## Kesimpulan

Sekarang pipeline kita sudah dapat:

- Crawl halaman
- Parse menjadi Markdown
- Chunking
- Generate embedding
- Melakukan semantic search

Tahap berikutnya adalah menggabungkan hasil retrieval ke dalam prompt LLM
untuk membangun sistem RAG sederhana.


In [7]:
conn.close()
print("Selesai 🎉")

Selesai 🎉
